# ♟️ chesSon: Supervised Learning Chess Engine Training

This notebook trains **chesSon**, a deep learning chess engine, using official Lichess broadcast tournament games (`.pgn.zst`).

### Key Highlights:
1. **Streaming Decompression**: Reads directly from `.pgn.zst` without exhausting Kaggle disk space.
2. **Quality Filtering**: Excludes games with < 15 moves or corrupt records, and trains exclusively on the winning moves.
3. **Canonical Tensor ($8 \times 8 \times 14$)**: Compact representation always oriented to the active player.
4. **Dual-Head ResNet**: Predicts policy (move selection, 4168 classes) and value (position evaluation in $[-1, 1]$).
5. **FP16 Mixed Precision**: Maximizes throughput on Kaggle GPUs (T4 / P100).

In [ ]:
# Step 1: Install required libraries
!pip install python-chess zstandard tqdm -q
print("Installed dependencies successfully!")

In [ ]:
# Step 2: Imports & Configuration
import io
import os
import sys
import time
import glob
import zstandard as zstd
import chess
import chess.pgn
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from tqdm import tqdm

# Search for your uploaded .pgn.zst dataset file automatically
found_files = glob.glob('/kaggle/input/**/*.pgn.zst', recursive=True)
default_path = found_files[0] if found_files else '/kaggle/input/lichess-broadcasts/broadcasts.pgn.zst'

CONFIG = {
    'pgn_zst_path': default_path,
    'batch_size': 512,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'num_res_blocks': 6,
    'channels': 128,
    'max_positions': 1_500_000,    # Target number of training positions
    'min_moves_per_game': 15,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'output_weights': '/kaggle/working/chesson_weights.pt',
}

print(f"Selected device: {CONFIG['device']}")
print(f"Target dataset path: {CONFIG['pgn_zst_path']}")

In [ ]:
# Step 3: Board & Move Encoding Definitions
ACTION_SIZE = 4168
NUM_PLANES = 14

PIECE_TO_PLANE = {
    chess.PAWN: 0,
    chess.KNIGHT: 1,
    chess.BISHOP: 2,
    chess.ROOK: 3,
    chess.QUEEN: 4,
    chess.KING: 5,
}

def encode_board(board: chess.Board) -> np.ndarray:
    planes = np.zeros((NUM_PLANES, 8, 8), dtype=np.float32)
    canonical = board.mirror() if board.turn == chess.BLACK else board

    for square in chess.SQUARES:
        piece = canonical.piece_at(square)
        if piece is not None:
            rank = chess.square_rank(square)
            file = chess.square_file(square)
            plane_idx = PIECE_TO_PLANE[piece.piece_type]
            if piece.color == chess.BLACK:
                plane_idx += 6
            planes[plane_idx, rank, file] = 1.0

    # Castling rights (Plane 12)
    if canonical.has_kingside_castling_rights(chess.WHITE):
        planes[12, 0, 4] = planes[12, 0, 7] = 1.0
    if canonical.has_queenside_castling_rights(chess.WHITE):
        planes[12, 0, 4] = planes[12, 0, 0] = 1.0
    if canonical.has_kingside_castling_rights(chess.BLACK):
        planes[12, 7, 4] = planes[12, 7, 7] = 1.0
    if canonical.has_queenside_castling_rights(chess.BLACK):
        planes[12, 7, 4] = planes[12, 7, 0] = 1.0

    # En passant (Plane 13)
    if canonical.ep_square is not None:
        planes[13, chess.square_rank(canonical.ep_square), chess.square_file(canonical.ep_square)] = 1.0

    return planes

def move_to_index(move: chess.Move, turn: chess.Color) -> int:
    if turn == chess.BLACK:
        from_sq = chess.square_mirror(move.from_square)
        to_sq = chess.square_mirror(move.to_square)
    else:
        from_sq = move.from_square
        to_sq = move.to_square

    if move.promotion is not None and move.promotion != chess.QUEEN:
        from_file = chess.square_file(from_sq)
        to_file = chess.square_file(to_sq)
        slot = (from_sq - 48) * 3 + (to_file - from_file + 1)
        if move.promotion == chess.KNIGHT:
            return 4096 + slot
        elif move.promotion == chess.BISHOP:
            return 4096 + 24 + slot
        elif move.promotion == chess.ROOK:
            return 4096 + 48 + slot

    return from_sq * 64 + to_sq

print("Board & move encoding ready.")

In [ ]:
# Step 4: Streaming Dataset from .pgn.zst
class StreamingChessDataset(IterableDataset):
    """Streams games directly from compressed .pgn.zst without uncompressing to disk."""

    def __init__(self, zst_path: str, max_positions: int = 1_000_000, min_moves: int = 15):
        self.zst_path = zst_path
        self.max_positions = max_positions
        self.min_moves = min_moves

    def __iter__(self):
        if not os.path.exists(self.zst_path):
            raise FileNotFoundError(f"Dataset file not found: {self.zst_path}")

        dctx = zstd.ZstdDecompressor()
        count = 0

        with open(self.zst_path, 'rb') as fh:
            with dctx.stream_reader(fh) as reader:
                text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')

                while count < self.max_positions:
                    try:
                        game = chess.pgn.read_game(text_stream)
                    except Exception:
                        continue

                    if game is None:
                        break

                    result = game.headers.get('Result', '*')
                    if result not in ('1-0', '0-1', '1/2-1/2'):
                        continue

                    moves = list(game.mainline_moves())
                    if len(moves) < self.min_moves:
                        continue

                    # Decide training targets based on winner
                    train_white = (result in ('1-0', '1/2-1/2'))
                    train_black = (result in ('0-1', '1/2-1/2'))

                    board = game.board()
                    for move in moves:
                        if not board.is_legal(move):
                            break

                        is_white_turn = (board.turn == chess.WHITE)
                        should_train = (is_white_turn and train_white) or (not is_white_turn and train_black)

                        if should_train:
                            if result == '1/2-1/2':
                                val_target = 0.0
                            elif (result == '1-0' and is_white_turn) or (result == '0-1' and not is_white_turn):
                                val_target = 1.0
                            else:
                                val_target = -1.0

                            board_tensor = encode_board(board)
                            move_idx = move_to_index(move, board.turn)

                            yield board_tensor, move_idx, np.float32(val_target)
                            count += 1
                            if count >= self.max_positions:
                                return

                        board.push(move)

print(f"StreamingChessDataset defined. Will stream up to {CONFIG['max_positions']:,} positions.")

In [ ]:
# Step 5: Model Definition (Dual-Headed ResNet with SiLU & GroupNorm)
class ResidualBlock(nn.Module):
    """Residual block with GroupNorm and SiLU activation."""
    def __init__(self, channels: int = 128):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(4, channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(4, channels),
        )
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(x + self.block(x))

class ChesSonNet(nn.Module):
    def __init__(self, num_blocks: int = 6, channels: int = 128):
        super().__init__()
        self.input = nn.Sequential(
            nn.Conv2d(NUM_PLANES, channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(4, channels),
            nn.SiLU(inplace=True),
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(channels) for _ in range(num_blocks)])
        self.policy = nn.Sequential(
            nn.Conv2d(channels, 64, kernel_size=1, bias=False),
            nn.GroupNorm(4, 64),
            nn.SiLU(inplace=True),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, ACTION_SIZE),
        )
        self.value = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=1, bias=False),
            nn.GroupNorm(4, 32),
            nn.SiLU(inplace=True),
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.SiLU(inplace=True),
            nn.Linear(128, 1),
            nn.Tanh(),
        )

    def forward(self, x: torch.Tensor):
        x = self.input(x)
        x = self.res_blocks(x)
        p_logits = self.policy(x)
        val = self.value(x)
        return p_logits, val

model = ChesSonNet(num_blocks=CONFIG['num_res_blocks'], channels=CONFIG['channels'])
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ChesSonNet created! Total trainable parameters: {total_params:,}")

In [ ]:
# Step 6: Training Execution with FP16 Mixed Precision
device = torch.device(CONFIG['device'])
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
policy_loss_fn = nn.CrossEntropyLoss()
value_loss_fn = nn.MSELoss()

dataset = StreamingChessDataset(
    zst_path=CONFIG['pgn_zst_path'],
    max_positions=CONFIG['max_positions'],
    min_moves=CONFIG['min_moves_per_game'],
)
dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'], num_workers=2, pin_memory=True)

print("Beginning training run...")
model.train()
total_loss, total_samples = 0.0, 0
correct_top1, correct_top3 = 0, 0
start_time = time.time()

pbar = tqdm(dataloader, desc="chesSon Training")
for batch_idx, (boards, moves, values) in enumerate(pbar):
    boards = boards.to(device, non_blocking=True)
    moves = moves.to(device, non_blocking=True)
    values = values.unsqueeze(1).to(device, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
        p_logits, v_pred = model(boards)
        loss_p = policy_loss_fn(p_logits, moves)
        loss_v = value_loss_fn(v_pred, values)
        loss = loss_p + 0.5 * loss_v

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    with torch.no_grad():
        _, top3 = p_logits.topk(3, dim=1)
        correct_top1 += (top3[:, 0] == moves).sum().item()
        correct_top3 += (top3 == moves.unsqueeze(1)).any(dim=1).sum().item()
        total_samples += moves.size(0)
        total_loss += loss.item()

    if (batch_idx + 1) % 50 == 0:
        pbar.set_postfix({
            'Loss': f'{total_loss / (batch_idx + 1):.3f}',
            'Top1%': f'{100.0 * correct_top1 / total_samples:.1f}%',
            'Top3%': f'{100.0 * correct_top3 / total_samples:.1f}%',
        })

elapsed = time.time() - start_time
print(f"\nTraining finished in {elapsed / 60:.2f} mins.")
print(f"Final Top-1 Accuracy: {100.0 * correct_top1 / total_samples:.2f}%")
print(f"Final Top-3 Accuracy: {100.0 * correct_top3 / total_samples:.2f}%")

In [ ]:
# Step 7: Export Model Weights
output_file = CONFIG['output_weights']
torch.save(model.state_dict(), output_file)
print(f"Weights successfully saved to: {output_file}")
print("Download this file from the Kaggle Output tab and place it into your local chesSon/weights/ folder!")